# Install Libraries

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers accelerate
!pip install torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.43.0 accelerate==0.31.0
!pip install bitsandbytes

Found existing installation: torch 2.3.1+cu121
Uninstalling torch-2.3.1+cu121:
  Successfully uninstalled torch-2.3.1+cu121
Found existing installation: torchvision 0.18.1+cu121
Uninstalling torchvision-0.18.1+cu121:
  Successfully uninstalled torchvision-0.18.1+cu121
Found existing installation: torchaudio 2.3.1+cu121
Uninstalling torchaudio-2.3.1+cu121:
  Successfully uninstalled torchaudio-2.3.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.3.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.9 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached https://downlo

# Create Config File

In [ ]:
yaml_config = """
model_path: "microsoft/Phi-3.5-mini-instruct"
max_new_tokens: 500
temperature: 0.7
top_p: 0.9
"""

with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# Build Model

In [2]:
import torch
from omegaconf import OmegaConf
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

class Phi3Chatbot:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_path,
            device_map="auto",
            torch_dtype="auto",
            quantization_config=bnb_config,
            trust_remote_code=True
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
        )

    def __call__(self, user_message):
        messages = [
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": user_message},
        ]

        generation_args = {
            "max_new_tokens": self.config.max_new_tokens,
            "return_full_text": False,
            "temperature": self.config.temperature,
            "top_p": self.config.top_p,
            "do_sample": True,
        }

        output = self.pipe(messages, **generation_args)
        return output[0]['generated_text']

Writing chatbot_logic.py


# Initialize Model

In [ ]:
chatbot = Phi3Chatbot("./config.yaml")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Initialize API

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import uvicorn
import threading

class ChatRequest(BaseModel):
    prompt: str

app = FastAPI()
nest_asyncio.apply()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/')
async def root():
    return {
        "system_name": "Hệ thống Chatbot Hướng dẫn du lịch Phi-3.5",
        "description": "API này sử dụng mô hình Phi-3.5-mini-instruct để tư vấn về du lịch Việt Nam."
    }

@app.get('/health')
async def health_check():
  try:
    is_alive= torch.cuda.is_available()
    return {
        "status": "ready" if is_alive else "error",
        "gpu_connected": is_alive,
        "model_name": "Phi-3.5-mini-instruct"
    }
  except:
      return {"status": "not ready", "reason": "Model not loaded"}

@app.post('/generate')
async def generate_api(request: ChatRequest):
    # Kiểm tra dữ liệu đầu vào cơ bản (Yêu cầu: Xử lý thiếu dữ liệu)
    if not request.prompt.strip():
        raise HTTPException(status_code=400, detail="Prompt không được để trống")
    try:
      response = chatbot(request.prompt)
      return {"status": "success", "response": response}
    except Exception as e:
        # Xử lý lỗi trong quá trình suy luận (Yêu cầu: Xử lý lỗi hợp lý)
        raise HTTPException(status_code=500, detail=f"Lỗi mô hình: {str(e)}")

# Hàm chạy server trong luồng riêng
def run_api():
    # Giải phóng cổng 8000 nếu bị kẹt
    !fuser -k 8000/tcp
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_api, daemon=True).start()

Writing main.py


# Call Local API

In [ ]:
import requests

res_root = requests.get(f"http://127.0.0.1:8000/")
print("Status:", res_root.status_code)
print("Kết quả:", res_root.json())

INFO:     127.0.0.1:34590 - "GET / HTTP/1.1" 200 OK
Status: 200
Kết quả: {'system_name': 'Hệ thống Chatbot Hướng dẫn du lịch Phi-3.5', 'description': 'API này sử dụng mô hình Phi-3.5-mini-instruct để tư vấn về du lịch Việt Nam.'}


In [ ]:
import requests

res_health = requests.get(f"http://127.0.0.1:8000/health")
print("Status:", res_health.status_code)
print("Kết quả:", res_health.json())

INFO:     127.0.0.1:49370 - "GET /health HTTP/1.1" 200 OK
Status: 200
Kết quả: {'status': 'ready', 'gpu_connected': True, 'model_name': 'Phi-3.5-mini-instruct'}


In [ ]:
import requests

url = "http://127.0.0.1:8000/generate"
data = {"prompt": "How is rice made?"}

response = requests.post(url, json=data)
result = response.json()
if response.status_code == 200:
    print("Kết quả từ API:", result.get("response"))
else:
    print("Lỗi Server:", result)

INFO:     127.0.0.1:56756 - "POST /generate HTTP/1.1" 200 OK
Kết quả từ API:  Rice is not made in the traditional sense but is cultivated and harvested from paddy fields. The process of rice cultivation involves several steps, from preparing the land to cooking the rice. Here's an overview:

1. **Land Preparation**: The first step is preparing the land where rice will be grown. Farmers typically plow the fields to loosen the soil, followed by harrowing to break up clods and ensure a smooth planting surface.

2. **Seedbed Preparation**: Rice is typically planted in flooded fields called paddies. Before planting, farmers prepare the seedbed by creating shallow trenches or channels, which will later be filled with water.

3. **Transplanting or Direct Sowing**:
   - **Transplanting**: In this method, farmers first grow rice seedlings in a separate nursery. Once the seedlings are robust (usually after about 30 days), they are carefully transplanted into the prepared paddy fields.
   - **Dir

# Call Public API

In [4]:
import requests

# 1. Địa chỉ API Public (Lấy từ link mà pinggy cung cấp khi chạy server)
# ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io
url = "https://ufvtz-35-230-50-130.run.pinggy-free.link/generate"

# 2. Dữ liệu gửi đi (Dạng Dictionary)
data = {"prompt": "How is rice made?"}

# 3. Thực hiện lệnh gọi POST
response = requests.post(url, json=data)

# 4. In kết quả JSON
print("Kết quả từ API:", response.json()["response"])

Writing test_api.py
